In [ ]:
import pickle
import torch

# Load the dataframe from the pickle file
with open('./df_input_scaling_bert_w2v2.pkl', 'rb') as file:
    df_input = pickle.load(file)

# Check the shape of the audio embeddings in train, valid, and test sets
train_audio_shape = df_input['train']['audio'].shape
valid_audio_shape = df_input['valid']['audio'].shape
test_audio_shape = df_input['test']['audio'].shape

# Print the shapes
print(f"Train audio embedding shape: {train_audio_shape}")
print(f"Valid audio embedding shape: {valid_audio_shape}")
print(f"Test audio embedding shape: {test_audio_shape}")

# Check the sequence length of a sample embedding
train_sample_seq_length = df_input['train']['audio'][0].shape
valid_sample_seq_length = df_input['valid']['audio'][0].shape
test_sample_seq_length = df_input['test']['audio'][0].shape

# Print the sequence lengths
print(f"Train sample sequence length: {train_sample_seq_length}")
print(f"Valid sample sequence length: {valid_sample_seq_length}")
print(f"Test sample sequence length: {test_sample_seq_length}")


In [ ]:
import numpy as np
import torch
import pickle
from tqdm.notebook import tqdm

# Function to average embeddings to a specific sequence length
def average_embeddings_to_length(embeddings, target_length):
    current_length = embeddings.shape[0]
    if current_length > target_length:
        # Calculate the step size for averaging
        step_size = current_length / target_length
        averaged_embeddings = []
        for i in range(target_length):
            start_idx = int(i * step_size)
            end_idx = int((i + 1) * step_size)
            # Ensure the indices are within bounds
            if end_idx > current_length:
                end_idx = current_length
            averaged_embeddings.append(embeddings[start_idx:end_idx].mean(axis=0))
        return np.array(averaged_embeddings)
    else:
        # If the current length is less than or equal to the target length, pad with zeros
        padding = np.zeros((target_length - current_length, embeddings.shape[1]))
        return np.concatenate([embeddings, padding], axis=0)

# Apply the function to the audio embeddings in the dataframe
target_length = 512  # The target sequence length to match the text embeddings

def apply_average_embeddings(df, target_length):
    new_embeddings = []
    for i in tqdm(range(df.shape[0]), desc="Processing embeddings"):
        new_embedding = average_embeddings_to_length(df[i].numpy(), target_length)
        new_embeddings.append(torch.tensor(new_embedding, dtype=torch.float32))
    return torch.stack(new_embeddings)

# Load the dataframe from the pickle file
with open('./df_input_scaling_bert_w2v2.pkl', 'rb') as file:
    df_input = pickle.load(file)

# Check the shapes before processing
print(f"Original Train audio shape: {df_input['train']['audio'].shape}")
print(f"Original Valid audio shape: {df_input['valid']['audio'].shape}")
print(f"Original Test audio shape: {df_input['test']['audio'].shape}")

# Apply the function to each dataset split
df_input['train']['audio'] = apply_average_embeddings(df_input['train']['audio'], target_length)
df_input['valid']['audio'] = apply_average_embeddings(df_input['valid']['audio'], target_length)
df_input['test']['audio'] = apply_average_embeddings(df_input['test']['audio'], target_length)

# Check the shapes after processing
print(f"Processed Train audio shape: {df_input['train']['audio'].shape}")
print(f"Processed Valid audio shape: {df_input['valid']['audio'].shape}")
print(f"Processed Test audio shape: {df_input['test']['audio'].shape}")

# Save the modified dataframe back to a new pickle file
with open('./df_input_scaling_bert_w2v2_reduced.pkl', 'wb') as file:
    pickle.dump(df_input, file)

print("Audio embeddings adjusted to target sequence length of 512 and saved successfully")
